# Chopin the Ⅲ ― 楽譜読み取りAIの学習（Google Colab 用）

このノートブックは、上から順に ▶（実行）を押していくだけで、楽譜読み取りAIの学習ができます。

**最初に1回だけ：GPU を選ぶ**
メニューの「ランタイム」→「ランタイムのタイプを変更」→「ハードウェア アクセラレータ」で **A100 GPU**（無ければ **L4 GPU**）を選んで「保存」。

**途中で止まっても大丈夫**
学習の途中経過は Google ドライブの「chopin-omr」フォルダに保存されます。止まったら、もう一度上から実行すると続きから再開します。

## 1. Google ドライブにつなぐ（保存先）
実行すると、Google アカウントの許可を求める画面が出ます。「許可」を押してください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE = '/content/drive/MyDrive/chopin-omr'   # 学習の結果を置く場所（変えなくてよい）
RUN = SAVE + '/run2'                          # 学習の名前（2版目）
PREV = SAVE + '/run1'                         # 1版目（ここから引き継ぐ）
os.makedirs(RUN, exist_ok=True)
print('保存先:', RUN)

## 2. GPU の確認
下に `A100` や `L4` と出れば準備できています。`NVIDIA-SMI has failed` と出たら、上の「最初に1回だけ」をやり直してください。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. プログラムを取ってくる・必要なものを入れる（1〜2分）

In [ ]:
%cd /content
!rm -rf /content/pkm && git clone -q --depth 1 https://github.com/hirosak514/pianokeymarker /content/pkm
%cd /content/pkm/omr-train
!pip install -q verovio cairosvg onnx onnxruntime rapidfuzz pymupdf
import torch; print('PyTorch', torch.__version__, '/ GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'なし')

## 4. 練習用の楽譜画像を作る
乱数で作った曲（著作権の問題が無い曲）を楽譜画像にし、写真のようなゆがみ・影・ぼけを加えます。
- `COUNT` が練習問題の数です。最初は **200000**（20万枚）がおすすめです。A100 の環境で 30〜60分ほどかかります。
- 作った画像は Google ドライブにも保存するので（約3GB）、2回目からは作り直さずにすぐ使えます。- **2版目**: 3連符・6連符、弱起、3つの声部、テンポ表示・ペダル・スタッカート・装飾音、隣の段の写り込みなどを加えた練習用の楽譜です。1版目の画像（`data_200000.zip`）は使わないので、ドライブの容量が足りなければ消して大丈夫です。


In [ ]:
# ── 準備（この欄だけ実行しても動くように、毎回確かめます）──
import os, sys
SAVE = '/content/drive/MyDrive/chopin-omr'; RUN = SAVE + '/run2'; PREV = SAVE + '/run1'  # run2 = 2版目（3連符などを覚える版）
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(RUN, exist_ok=True)
if not os.path.exists('/content/pkm/omr-train/omr'):
    print('プログラムを取ってきます…')
    os.chdir('/content')
    os.system('rm -rf /content/pkm && git clone -q --depth 1 https://github.com/hirosak514/pianokeymarker /content/pkm')
    os.system('pip install -q verovio cairosvg onnx onnxruntime rapidfuzz pymupdf')
os.chdir('/content/pkm/omr-train')
# ──────────────────────────────────────────
COUNT = 200000
import os, subprocess
DATA = '/content/data2'
ZIP = SAVE + f'/data_v2_{COUNT}.zip'  # 2版目の練習用画像（1版目の data_200000.zip とは別）
if os.path.exists(DATA + '/train/labels.tsv'):
    print('作ってある画像を使います')
elif os.path.exists(ZIP) and os.path.getsize(ZIP) > 10**8:
    print('ドライブに保存してある画像を取り出します…')
    !mkdir -p /content/data2 && cd /content/data2 && unzip -q -o "$ZIP"
if not os.path.exists(DATA + '/train/labels.tsv'):
    n = os.cpu_count()
    print(f'CPU {n}個で作ります')
    !python -u -m omr.make_data --out /content/data2/val --count 2000 --seed 999 --workers {n}
    !python -u -m omr.make_data --out /content/data2/train --count {COUNT} --workers {n}
    print('ドライブに保存しています…（途中で切れても壊れたファイルが残らないよう、いったん別名で保存します）')
    !cd /content/data2 && rm -f /content/data2.zip && zip -q -r -0 /content/data2.zip train val && cp /content/data2.zip "$ZIP.tmp" && mv "$ZIP.tmp" "$ZIP" && rm -f /content/data2.zip
    print('保存しました:', ZIP if os.path.exists(ZIP) else '（保存に失敗しました。ドライブの空き容量を確かめてください）')
!wc -l /content/data2/train/labels.tsv /content/data2/val/labels.tsv

## 4b. 本物の楽譜を学習に加える（任意・GPU が無くても動きます）
本物の楽譜の画像と、その曲の MusicXML を組にして学習に使います。練習用の楽譜だけで学ぶより、本物の印刷・写真に強くなります。

**準備（Google ドライブで）**
1. ドライブの `chopin-omr` フォルダの中に `real` というフォルダを作る
2. その中に、**1曲ごとに1つのフォルダ**を作る（名前は自由。例: `エリーゼのために`）
3. 曲のフォルダに、次の2種類を入れる
   - その曲の **MusicXML**（`.musicxml` / `.xml` / `.mxl` のどれか1つ）
   - その曲の **楽譜の画像**（`.jpg` `.png` など。ページの順に名前を付ける: `1.jpg` `2.jpg` …）または **PDF**

```
chopin-omr/real/エリーゼのために/score.mxl
chopin-omr/real/エリーゼのために/1.jpg
chopin-omr/real/エリーゼのために/2.jpg
chopin-omr/real/別れの曲/score.musicxml
chopin-omr/real/別れの曲/score.pdf
```
画像の版と MusicXML の段の区切り（1段に何小節あるか）は違っていて大丈夫です。今の AI で各段を試しに読んで、どの小節がどの段に載っているかを自動で合わせます。
合わせられなかった段や、5連符など（まだ書けない記号）を含む段は使いません。結果の表で「○」が使う段です。

**注意:** 著作権の切れた古い楽譜や、ご自分で作った楽譜を使ってください。

曲を足したら、この欄をもう一度実行してから、手順5を実行してください。

In [ ]:
# ── 準備（この欄だけ実行しても動くように、毎回確かめます）──
import os, sys
SAVE = '/content/drive/MyDrive/chopin-omr'; RUN = SAVE + '/run2'; PREV = SAVE + '/run1'  # run2 = 2版目（3連符などを覚える版）
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(RUN, exist_ok=True)
if not os.path.exists('/content/pkm/omr-train/omr'):
    print('プログラムを取ってきます…')
    os.chdir('/content')
    os.system('rm -rf /content/pkm && git clone -q --depth 1 https://github.com/hirosak514/pianokeymarker /content/pkm')
    os.system('pip install -q verovio cairosvg onnx onnxruntime rapidfuzz pymupdf')
os.chdir('/content/pkm/omr-train')
# ──────────────────────────────────────────
REAL_SRC = SAVE + '/real'
REAL = SAVE + '/real_data'
if not os.path.isdir(REAL_SRC):
    os.makedirs(REAL_SRC, exist_ok=True)
    print('ドライブに chopin-omr/real フォルダを作りました。上の説明のとおり曲のフォルダを入れてから、もう一度実行してください')
else:
    CK = next((c for c in (RUN + '/best.pt', RUN + '/last.pt', PREV + '/best.pt') if os.path.exists(c)), None)
    !rm -rf "$REAL" && python -u -m omr.realdata --src "$REAL_SRC" --out "$REAL" --ckpt "$CK"
    import pandas as pd
    if os.path.exists(REAL + '/report.tsv'):
        display(pd.read_csv(REAL + '/report.tsv', sep='\t'))


## 5. 学習する（いちばん時間がかかります）
- `HOURS` の時間だけ学習して止まります（Colab の1回の利用時間に合わせて、9時間にしています）。
- 2000歩ごとに「検証: 記号の間違い ○%」と出ます。この数字が小さくなっていけば、うまく覚えています。
- 止まったら（時間切れ・切断など）、もう一度 **1 から順に** 実行すれば、続きから再開します。
- 手順4bで本物の楽譜を加えたときは、「本物の楽譜: 記号の間違い ○%」も出ます。こちらが実際の読み取りの良さに近い数字で、いちばん良かった時点を best.pt に残します。
- **2版目**（`run2` フォルダ）は、1版目（`run1`）で覚えたことを引き継いで始めます。最初の検証は1版目より悪く出ますが（3連符などを知らないため）、だんだん下がります。8万歩で止まります。


In [ ]:
# ── 準備（この欄だけ実行しても動くように、毎回確かめます）──
import os, sys
SAVE = '/content/drive/MyDrive/chopin-omr'; RUN = SAVE + '/run2'; PREV = SAVE + '/run1'  # run2 = 2版目（3連符などを覚える版）
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(RUN, exist_ok=True)
if not os.path.exists('/content/pkm/omr-train/omr'):
    print('プログラムを取ってきます…')
    os.chdir('/content')
    os.system('rm -rf /content/pkm && git clone -q --depth 1 https://github.com/hirosak514/pianokeymarker /content/pkm')
    os.system('pip install -q verovio cairosvg onnx onnxruntime rapidfuzz pymupdf')
os.chdir('/content/pkm/omr-train')
# ──────────────────────────────────────────
HOURS = 9
import os
REAL = SAVE + '/real_data'
EXTRA = f'--extra "{REAL}/train" --val-real "{REAL}/val"' if os.path.exists(REAL + '/train/labels.tsv') else ''
print('本物の楽譜も使います' if EXTRA else '練習用の楽譜だけで学習します（本物の楽譜を加えるには手順4b）')
INIT = f'--init-from "{PREV}/best.pt"' if os.path.exists(PREV + '/best.pt') else ''
!python -u -m omr.train --data /content/data2/train --val /content/data2/val --out "$RUN" --hours {HOURS} --workers {os.cpu_count()} --max-steps 80000 --warmup 1000 --lr 2e-4 {INIT} {EXTRA}

## 6. 学習の進み具合をグラフで見る
左のグラフ（記号の間違いの割合）が下がり、右のグラフ（1段を完全に正しく読めた割合）が上がっていれば順調です。

In [ ]:
# ── 準備（この欄だけ実行しても動くように、毎回確かめます）──
import os, sys
SAVE = '/content/drive/MyDrive/chopin-omr'; RUN = SAVE + '/run2'; PREV = SAVE + '/run1'  # run2 = 2版目（3連符などを覚える版）
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(RUN, exist_ok=True)
if not os.path.exists('/content/pkm/omr-train/omr'):
    print('プログラムを取ってきます…')
    os.chdir('/content')
    os.system('rm -rf /content/pkm && git clone -q --depth 1 https://github.com/hirosak514/pianokeymarker /content/pkm')
    os.system('pip install -q verovio cairosvg onnx onnxruntime rapidfuzz pymupdf')
os.chdir('/content/pkm/omr-train')
# ──────────────────────────────────────────
import json, matplotlib.pyplot as plt
rows = [json.loads(l) for l in open(RUN + '/log.jsonl')]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot([r['step'] for r in rows], [r['ser'] * 100 for r in rows]); ax[0].set_title('symbol error % (lower is better)'); ax[0].set_xlabel('step')
ax[1].plot([r['step'] for r in rows], [r['exact'] * 100 for r in rows]); ax[1].set_title('systems read perfectly %'); ax[1].set_xlabel('step')
rr = [r for r in rows if 'real_ser' in r]
if rr:
    ax[0].plot([r['step'] for r in rr], [r['real_ser'] * 100 for r in rr], label='real scores'); ax[0].legend()
    ax[1].plot([r['step'] for r in rr], [r['real_exact'] * 100 for r in rr], label='real scores'); ax[1].legend()
plt.show()
print('最新:', rows[-1])

## 7. アプリで使う形（ONNX）に書き出す
ドライブの `chopin-omr/run2/onnx` フォルダに `encoder.onnx`・`decoder.onnx`・`vocab.json` ができます。
**この3つのファイル**をチャットに添付してください（アプリに組み込みます）。

In [ ]:
# ── 準備（この欄だけ実行しても動くように、毎回確かめます）──
import os, sys
SAVE = '/content/drive/MyDrive/chopin-omr'; RUN = SAVE + '/run2'; PREV = SAVE + '/run1'  # run2 = 2版目（3連符などを覚える版）
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(RUN, exist_ok=True)
if not os.path.exists('/content/pkm/omr-train/omr'):
    print('プログラムを取ってきます…')
    os.chdir('/content')
    os.system('rm -rf /content/pkm && git clone -q --depth 1 https://github.com/hirosak514/pianokeymarker /content/pkm')
    os.system('pip install -q verovio cairosvg onnx onnxruntime rapidfuzz pymupdf')
os.chdir('/content/pkm/omr-train')
# ──────────────────────────────────────────
!python -m omr.export --ckpt "$RUN/best.pt" --out "$RUN/onnx"
!ls -la "$RUN/onnx"

## 8. 自分の楽譜で試す（任意・GPU が無くても動きます）
ページ全体の画像・写真でも大丈夫です。大譜表を1段ずつ自動で切り出して読み、切り出した画像も下に表示します。


In [ ]:
# ── 準備（この欄だけ実行しても動くように、毎回確かめます）──
import os, sys
SAVE = '/content/drive/MyDrive/chopin-omr'; RUN = SAVE + '/run2'; PREV = SAVE + '/run1'  # run2 = 2版目（3連符などを覚える版）
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(RUN, exist_ok=True)
if not os.path.exists('/content/pkm/omr-train/omr'):
    print('プログラムを取ってきます…')
    os.chdir('/content')
    os.system('rm -rf /content/pkm && git clone -q --depth 1 https://github.com/hirosak514/pianokeymarker /content/pkm')
    os.system('pip install -q verovio cairosvg onnx onnxruntime rapidfuzz pymupdf')
os.chdir('/content/pkm/omr-train')
# ──────────────────────────────────────────
from google.colab import files
up = files.upload()
for name in up:
    !python -m omr.predict --ckpt "$RUN/best.pt" "$name"
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('crops/*.png'))[:12]:
    print(f); display(Image(f))